# Exploratory Analysis of New York City Airbnb Listings

**Author:** Marius Jochheim  
**Dataset:** New York City Airbnb Open Data (2019)

This project develops a reproducible Python workflow for cleaning, exploring,
and visualizing New York City Airbnb listing data. The analysis focuses on how
listing prices and availability differ across neighbourhood groups and room
types.

## Analysis Questions

This analysis investigates the following questions:

1. How do Airbnb listing prices vary across New York City neighbourhood groups?
2. How do prices differ between room types?
3. How does listing availability vary across neighbourhood groups and room types?
4. Are price, review activity, minimum stay, and availability meaningfully related?

## Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Load Dataset

In [2]:
DATA_PATH = "dataset/AB_NYC_2019.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [3]:
print(f"The dataset contains {df.shape[0]:,} rows and {df.shape[1]} columns.")

The dataset contains 48,895 rows and 16 columns.


## Initial Data Inspection

Before cleaning the dataset, its structure, data types, missing values,
duplicates, and numerical distributions are inspected.

### Columns, missing values and duplicates

In [4]:
def inspect_dataset(data):
    """
    Display a concise overview of a pandas DataFrame.

    The function reports the dataset dimensions, data types, missing-value
    counts, duplicated rows, and descriptive statistics for numerical columns.

    Parameters
    ----------
    data : pandas.DataFrame
        The dataset to inspect.

    Returns
    -------
    None
        The function prints and displays inspection results.
    """
    print(f"Rows: {data.shape[0]:,}")
    print(f"Columns: {data.shape[1]}")
    print(f"Duplicated rows: {data.duplicated().sum():,}")

    display(
        data.dtypes
        .astype(str)
        .rename("data_type")
        .to_frame()
    )

    missing_summary = (
        df.isna()
        .sum()
        .rename("missing_count")
        .to_frame()
    )

    missing_summary["missing_percentage"] = (
        missing_summary["missing_count"] / len(df) * 100
    )

    missing_summary = missing_summary.sort_values(
        by="missing_count",
        ascending=False
    )

    display(
        missing_summary
    )

    display(data.describe(include="number").T)

In [5]:
inspect_dataset(df)

Rows: 48,895
Columns: 16
Duplicated rows: 0


,data_type
id,int64
name,str
host_id,int64
host_name,str
neighbourhood_group,str
neighbourhood,str
latitude,float64
longitude,float64
room_type,str
price,int64


,missing_count,missing_percentage
last_review,10052,20.558339
reviews_per_month,10052,20.558339
host_name,21,0.042949
name,16,0.032723
neighbourhood_group,0,0.000000
neighbourhood,0,0.000000
id,0,0.000000
host_id,0,0.000000
longitude,0,0.000000
latitude,0,0.000000


,count,mean,std,min,25%,50%,75%,max
id,48895.0,1.901714e+07,1.098311e+07,2539.00000,9.471945e+06,1.967728e+07,2.915218e+07,3.648724e+07
host_id,48895.0,6.762001e+07,7.861097e+07,2438.00000,7.822033e+06,3.079382e+07,1.074344e+08,2.743213e+08
latitude,48895.0,4.072895e+01,5.453008e-02,40.49979,4.069010e+01,4.072307e+01,4.076311e+01,4.091306e+01
longitude,48895.0,-7.395217e+01,4.615674e-02,-74.24442,-7.398307e+01,-7.395568e+01,-7.393627e+01,-7.371299e+01
price,48895.0,1.527207e+02,2.401542e+02,0.00000,6.900000e+01,1.060000e+02,1.750000e+02,1.000000e+04
minimum_nights,48895.0,7.029962e+00,2.051055e+01,1.00000,1.000000e+00,3.000000e+00,5.000000e+00,1.250000e+03
number_of_reviews,48895.0,2.327447e+01,4.455058e+01,0.00000,1.000000e+00,5.000000e+00,2.400000e+01,6.290000e+02
reviews_per_month,38843.0,1.373221e+00,1.680442e+00,0.01000,1.900000e-01,7.200000e-01,2.020000e+00,5.850000e+01
calculated_host_listings_count,48895.0,7.143982e+00,3.295252e+01,1.00000,1.000000e+00,1.000000e+00,2.000000e+00,3.270000e+02
availability_365,48895.0,1.127813e+02,1.316223e+02,0.00000,0.000000e+00,4.500000e+01,2.270000e+02,3.650000e+02


### Categorical Values

In [6]:
for column in ["neighbourhood_group", "room_type"]:
    print(f"\nValues in {column}:")
    print(df[column].value_counts(dropna=False))


Values in neighbourhood_group:
neighbourhood_group
Manhattan        21661
Brooklyn         20104
Queens            5666
Bronx             1091
Staten Island      373
Name: count, dtype: int64

Values in room_type:
room_type
Entire home/apt    25409
Private room       22326
Shared room         1160
Name: count, dtype: int64


In [7]:
categorical_summary = pd.DataFrame({
    "unique_values": df[
        ["neighbourhood_group", "neighbourhood", "room_type"]
    ].nunique(),
    "missing_values": df[
        ["neighbourhood_group", "neighbourhood", "room_type"]
    ].isna().sum()
})

categorical_summary

,unique_values,missing_values
neighbourhood_group,5,0
neighbourhood,221,0
room_type,3,0


### Numerical Ranges

In [8]:
analysis_columns = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365"
]

df[analysis_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
price,48895.0,152.720687,240.154170,0.00,69.00,106.00,175.00,10000.0
minimum_nights,48895.0,7.029962,20.510550,1.00,1.00,3.00,5.00,1250.0
number_of_reviews,48895.0,23.274466,44.550582,0.00,1.00,5.00,24.00,629.0
reviews_per_month,38843.0,1.373221,1.680442,0.01,0.19,0.72,2.02,58.5
calculated_host_listings_count,48895.0,7.143982,32.952519,1.00,1.00,1.00,2.00,327.0
availability_365,48895.0,112.781327,131.622289,0.00,0.00,45.00,227.00,365.0


### Initial Inspection Observations

- The dataset contains **48,895 rows** and **16 columns**.
- Each row represents an Airbnb listing in New York City.
- Missing values occur primarily in `last_review`	and `reviews_per_month`.
- The `last_review` column is currently stored as text and should be converted
  to a date type.
- The complete dataset contains **no duplicated rows**.
- The price and minimum-night variables contain large values that require
  further investigation before deciding whether they should be retained,
  filtered, or treated as outliers.

## Data Cleaning

The cleaning process is divided into reusable functions. Review-related missing
values are handled according to their context, while rows are removed only when values are invalid for the planned analysis.

Extreme but technically possible values are retained initially and examined
separately during exploratory analysis.

In [9]:
def clean_review_data(data):
    """
    Clean date and review-related fields in an Airbnb listings dataset.

    The function converts `last_review` to a pandas datetime column. Missing
    `reviews_per_month` values are replaced with zero only for listings whose
    `number_of_reviews` is zero.

    Parameters
    ----------
    data : pandas.DataFrame
        Airbnb listings data containing `last_review`, `reviews_per_month`,
        and `number_of_reviews`.

    Returns
    -------
    pandas.DataFrame
        A copy of the dataset with cleaned review-related fields.

    Raises
    ------
    KeyError
        If one or more required columns are missing.
    """
    required_columns = {
        "last_review",
        "reviews_per_month",
        "number_of_reviews",
    }

    missing_columns = required_columns.difference(data.columns)

    if missing_columns:
        raise KeyError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    cleaned_data = data.copy()

    cleaned_data["last_review"] = pd.to_datetime(
        cleaned_data["last_review"],
        errors="coerce",
    )

    zero_review_mask = (
        cleaned_data["number_of_reviews"].eq(0)
        & cleaned_data["reviews_per_month"].isna()
    )

    cleaned_data.loc[
        zero_review_mask,
        "reviews_per_month",
    ] = 0.0

    return cleaned_data

In [10]:
def remove_invalid_records(data):
    """
    Remove duplicate listings and records with invalid analytical values.

    The function removes listings with non-positive prices, 
    and listings with non-positive minimum-night requirements. 
    It also removes records whose availability is outside the
    expected range of 0 to 365 days.

    Parameters
    ----------
    data : pandas.DataFrame
        Airbnb listings data containing `id`, `price`, `minimum_nights`,
        and `availability_365`.

    Returns
    -------
    pandas.DataFrame
        A cleaned copy of the dataset with its index reset.

    Raises
    ------
    KeyError
        If one or more required columns are missing.
    """
    required_columns = {
        "id",
        "price",
        "minimum_nights",
        "availability_365",
    }

    missing_columns = required_columns.difference(data.columns)

    if missing_columns:
        raise KeyError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    cleaned_data = data.copy()

    valid_record_mask = (
        cleaned_data["price"].gt(0)
        & cleaned_data["minimum_nights"].gt(0)
        & cleaned_data["availability_365"].between(0, 365)
    )

    cleaned_data = cleaned_data.loc[valid_record_mask]

    return cleaned_data.reset_index(drop=True)

In [11]:
def summarize_cleaning(original_data, cleaned_data):
    """
    Summarize row counts and selected validation checks before and after cleaning.

    Parameters
    ----------
    original_data : pandas.DataFrame
        Dataset before cleaning.
    cleaned_data : pandas.DataFrame
        Dataset after cleaning.

    Returns
    -------
    pandas.DataFrame
        A table comparing dataset size, duplicates, and missing values.
    """
    return pd.DataFrame({
        "original": {
            "rows": len(original_data),
            "duplicate_rows": original_data.duplicated().sum(),
            "duplicate_ids": original_data["id"].duplicated().sum(),
            "missing_values": original_data.isna().sum().sum(),
            "non_positive_prices": original_data["price"].le(0).sum(),
        },
        "cleaned": {
            "rows": len(cleaned_data),
            "duplicate_rows": cleaned_data.duplicated().sum(),
            "duplicate_ids": cleaned_data["id"].duplicated().sum(),
            "missing_values": cleaned_data.isna().sum().sum(),
            "non_positive_prices": cleaned_data["price"].le(0).sum(),
        },
    })

In [12]:
df_clean = clean_review_data(df)
df_clean = remove_invalid_records(df_clean)
summarize_cleaning(df, df_clean)

,original,cleaned
rows,48895,48884
duplicate_rows,0,0
duplicate_ids,0,0
missing_values,20141,10088
non_positive_prices,11,0


In [13]:
# Check that the cleaning process worked

cleaning_checks = pd.Series({
    "duplicate_rows": df_clean.duplicated().sum(),
    "duplicate_listing_ids": df_clean["id"].duplicated().sum(),
    "non_positive_prices": df_clean["price"].le(0).sum(),
    "non_positive_minimum_nights": (
        df_clean["minimum_nights"].le(0).sum()
    ),
    "availability_outside_range": (
        ~df_clean["availability_365"].between(0, 365)
    ).sum(),
    "zero_reviews_with_missing_rate": (
        df_clean["number_of_reviews"].eq(0)
        & df_clean["reviews_per_month"].isna()
    ).sum(),
})

cleaning_checks

duplicate_rows                    0
duplicate_listing_ids             0
non_positive_prices               0
non_positive_minimum_nights       0
availability_outside_range        0
zero_reviews_with_missing_rate    0
dtype: int64

In [14]:
# check that the date conversion of last_review worked

print("Original type:", df["last_review"].dtype)
print("Cleaned type:", df_clean["last_review"].dtype)

Original type: str
Cleaned type: datetime64[us]


In [15]:
df_clean[
    [
        "number_of_reviews",
        "last_review",
        "reviews_per_month",
    ]
].head(10)

,number_of_reviews,last_review,reviews_per_month
0,9,2018-10-19,0.21
1,45,2019-05-21,0.38
2,0,NaT,0.00
3,270,2019-07-05,4.64
4,9,2018-11-19,0.10
5,74,2019-06-22,0.59
6,49,2017-10-05,0.40
7,430,2019-06-24,3.47
8,118,2017-07-21,0.99
9,160,2019-06-09,1.33


In [16]:
# Deep dive into remaining missing values

remaining_missing = (
    df_clean.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

remaining_missing["missing_percentage"] = (
    remaining_missing["missing_count"]
    / len(df_clean)
    * 100
)

remaining_missing = remaining_missing.loc[
    remaining_missing["missing_count"].gt(0)
].sort_values(
    "missing_count",
    ascending=False,
)

remaining_missing

,missing_count,missing_percentage
last_review,10051,20.560920
host_name,21,0.042959
name,16,0.032731


### Cleaning Decisions

The cleaning functions removed **11 records**, reducing the dataset from
**48,895** to **48,884** listings. Non-positive prices, non-positive minimum-night
values, and availability values outside the range of 0 to 365 were treated as
invalid.

The `last_review` column was converted from text to a date type. Missing
`reviews_per_month` values were replaced with zero only when the corresponding
listing had zero recorded reviews. Missing review dates were retained because
a listing without reviews does not have a meaningful last-review date.

Missing listing and host names were also retained because these descriptive
fields are not required for the planned price and availability analysis.
Potential price and minimum-night outliers were not automatically removed,
because unusually high values may represent valid listings rather than data
entry errors.

## Exploratory Data Analysis

The exploratory analysis summarizes listing prices, availability, review
activity, and minimum-stay requirements. Grouped comparisons are used to
examine differences between neighbourhood groups and room types.

### Create exploratory summary tables

In [17]:
def explore_airbnb_data(data):
    """
    Generate exploratory summary tables for Airbnb listing data.

    The function returns:
    1. Overall descriptive statistics for selected numerical variables.
    2. Listing counts and price statistics by neighbourhood group.
    3. Listing counts and price statistics by room type.
    4. Price and availability statistics grouped by neighbourhood group
       and room type.
    5. A correlation matrix for selected numerical variables.

    Parameters
    ----------
    data : pandas.DataFrame
        Cleaned Airbnb listing data.

    Returns
    -------
    dict
        A dictionary containing the exploratory summary tables.

    Raises
    ------
    KeyError
        If required columns are missing from the dataset.
    """
    required_columns = {
        "price",
        "minimum_nights",
        "number_of_reviews",
        "reviews_per_month",
        "availability_365",
        "calculated_host_listings_count",
        "neighbourhood_group",
        "room_type",
    }

    missing_columns = required_columns.difference(data.columns)

    if missing_columns:
        raise KeyError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    numerical_columns = [
        "price",
        "minimum_nights",
        "number_of_reviews",
        "reviews_per_month",
        "calculated_host_listings_count",
        "availability_365",
    ]

    overall_summary = data[numerical_columns].describe().T

    neighbourhood_summary = (
        data.groupby("neighbourhood_group", observed=True)
        .agg(
            listing_count=("id", "count"),
            median_price=("price", "median"),
            mean_price=("price", "mean"),
            minimum_price=("price", "min"),
            maximum_price=("price", "max"),
            median_availability=("availability_365", "median"),
            median_reviews=("number_of_reviews", "median"),
        )
        .sort_values("median_price", ascending=False)
    )

    room_type_summary = (
        data.groupby("room_type", observed=True)
        .agg(
            listing_count=("id", "count"),
            median_price=("price", "median"),
            mean_price=("price", "mean"),
            minimum_price=("price", "min"),
            maximum_price=("price", "max"),
            median_availability=("availability_365", "median"),
            median_reviews=("number_of_reviews", "median"),
        )
        .sort_values("median_price", ascending=False)
    )

    grouped_summary = (
        data.groupby(
            ["neighbourhood_group", "room_type"],
            observed=True,
        )
        .agg(
            listing_count=("id", "count"),
            median_price=("price", "median"),
            mean_price=("price", "mean"),
            median_availability=("availability_365", "median"),
            median_minimum_nights=("minimum_nights", "median"),
            median_reviews=("number_of_reviews", "median"),
        )
        .sort_values(
            ["neighbourhood_group", "median_price"],
            ascending=[True, False],
        )
    )

    correlation_matrix = data[numerical_columns].corr(
        method="pearson"
    )

    return {
        "overall_summary": overall_summary,
        "neighbourhood_summary": neighbourhood_summary,
        "room_type_summary": room_type_summary,
        "grouped_summary": grouped_summary,
        "correlation_matrix": correlation_matrix,
    }

In [18]:
eda_results = explore_airbnb_data(df_clean)

In [19]:
eda_results["overall_summary"]

,count,mean,std,min,25%,50%,75%,max
price,48884.0,152.755053,240.170260,10.0,69.00,106.00,175.00,10000.0
minimum_nights,48884.0,7.029887,20.512224,1.0,1.00,3.00,5.00,1250.0
number_of_reviews,48884.0,23.271991,44.551331,0.0,1.00,5.00,24.00,629.0
reviews_per_month,48884.0,1.090800,1.597213,0.0,0.04,0.37,1.58,58.5
calculated_host_listings_count,48884.0,7.144628,32.956185,1.0,1.00,1.00,2.00,327.0
availability_365,48884.0,112.779498,131.627271,0.0,0.00,45.00,227.00,365.0


In [20]:
eda_results["neighbourhood_summary"]

,listing_count,median_price,mean_price,minimum_price,maximum_price,median_availability,median_reviews
neighbourhood_group,,,,,,,
Manhattan,21660,150.0,196.884903,10,10000,36.0,4.0
Brooklyn,20095,90.0,124.438915,10,10000,28.0,6.0
Queens,5666,75.0,99.517649,10,10000,98.0,7.0
Staten Island,373,75.0,114.812332,13,5000,219.0,12.0
Bronx,1090,65.0,87.577064,10,2500,148.0,9.0


In [21]:
eda_results["room_type_summary"]

,listing_count,median_price,mean_price,minimum_price,maximum_price,median_availability,median_reviews
room_type,,,,,,,
Entire home/apt,25407,160.0,211.810918,10,10000,42.0,5.0
Private room,22319,70.0,89.809131,10,10000,45.0,5.0
Shared room,1158,45.0,70.248705,10,1800,90.0,4.0


In [22]:
eda_results["grouped_summary"]

listing_count  median_price  mean_price  \
neighbourhood_group room_type                                                  
Bronx               Entire home/apt            379         100.0  127.506596   
                    Private room               651          54.0   66.890937   
                    Shared room                 60          40.0   59.800000   
Brooklyn            Entire home/apt           9558         145.0  178.346202   
                    Private room             10126          65.0   76.545428   
                    Shared room                411          36.0   50.773723   
Manhattan           Entire home/apt          13198         191.0  249.257994   
                    Private room              7982          90.0  116.776622   
                    Shared room                480          69.0   88.977083   
Queens              Entire home/apt           2096         120.0  147.050573   
                    Private room              3372          60.0   71.762456   
                    Shared room                198          37.0   69.020202   
Staten Island       Entire home/apt            176         100.0  173.846591   
                    Private room               188          50.0   62.292553   
                    Shared room                  9          30.0   57.444444   

                                     median_availability  \
neighbourhood_group room_type                              
Bronx               Entire home/apt                131.0   
                    Private room                   158.0   
                    Shared room                     89.0   
Brooklyn            Entire home/apt                 28.0   
                    Private room                    24.0   
                    Shared room                    156.0   
Manhattan           Entire home/apt                 42.0   
                    Private room                    29.0   
                    Shared room                     81.0   
Queens              Entire home/apt                 89.0   
                    Private room                   108.0   
                    Shared room                    175.5   
Staten Island       Entire home/apt                176.5   
                    Private room                   282.0   
                    Shared room                     37.0   

                                     median_minimum_nights  median_reviews  
neighbourhood_group room_type                                               
Bronx               Entire home/apt                    2.0            11.0  
                    Private room                       2.0             9.0  
                    Shared room                        1.0             3.0  
Brooklyn            Entire home/apt                    3.0             7.0  
                    Private room                       2.0             4.0  
                    Shared room                        2.0             3.0  
Manhattan           Entire home/apt                    3.0             4.0  
                    Private room                       2.0             5.0  
                    Shared room                        1.0             6.0  
Queens              Entire home/apt                    2.0             8.0  
                    Private room                       2.0             7.0  
                    Shared room                        1.0             3.0  
Staten Island       Entire home/apt                    2.0            15.0  
                    Private room                       2.0            12.0  
                    Shared room                        2.0             1.0

In [23]:
eda_results["correlation_matrix"].round(2)

,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
price,1.00,0.04,-0.05,-0.05,0.06,0.08
minimum_nights,0.04,1.00,-0.08,-0.12,0.13,0.14
number_of_reviews,-0.05,-0.08,1.00,0.59,-0.07,0.17
reviews_per_month,-0.05,-0.12,0.59,1.00,-0.05,0.16
calculated_host_listings_count,0.06,0.13,-0.07,-0.05,1.00,0.23
availability_365,0.08,0.14,0.17,0.16,0.23,1.00


### Create Summaries for Neighbourhoods and Room types

In [24]:
neighbourhood_summary = (
    eda_results["neighbourhood_summary"].copy()
)

neighbourhood_summary["listing_percentage"] = (
    neighbourhood_summary["listing_count"]
    / len(df_clean)
    * 100
)

neighbourhood_summary[
    [
        "listing_count",
        "listing_percentage",
        "median_price",
        "mean_price",
        "median_availability",
    ]
].round(2)

,listing_count,listing_percentage,median_price,mean_price,median_availability
neighbourhood_group,,,,,
Manhattan,21660,44.31,150.0,196.88,36.0
Brooklyn,20095,41.11,90.0,124.44,28.0
Queens,5666,11.59,75.0,99.52,98.0
Staten Island,373,0.76,75.0,114.81,219.0
Bronx,1090,2.23,65.0,87.58,148.0


In [25]:
room_type_summary = eda_results["room_type_summary"].copy()

room_type_summary["listing_percentage"] = (
    room_type_summary["listing_count"]
    / len(df_clean)
    * 100
)

room_type_summary[
    [
        "listing_count",
        "listing_percentage",
        "median_price",
        "mean_price",
        "median_availability",
    ]
].round(2)

,listing_count,listing_percentage,median_price,mean_price,median_availability
room_type,,,,,
Entire home/apt,25407,51.97,160.0,211.81,42.0
Private room,22319,45.66,70.0,89.81,45.0
Shared room,1158,2.37,45.0,70.25,90.0


In [26]:
highest_price_neighbourhood = (
    neighbourhood_summary["median_price"].idxmax()
)

lowest_price_neighbourhood = (
    neighbourhood_summary["median_price"].idxmin()
)

highest_price_room_type = (
    room_type_summary["median_price"].idxmax()
)

lowest_price_room_type = (
    room_type_summary["median_price"].idxmin()
)

print(
    "Highest median-price neighbourhood group:",
    highest_price_neighbourhood,
)

print(
    "Lowest median-price neighbourhood group:",
    lowest_price_neighbourhood,
)

print(
    "Highest median-price room type:",
    highest_price_room_type,
)

print(
    "Lowest median-price room type:",
    lowest_price_room_type,
)

Highest median-price neighbourhood group: Manhattan
Lowest median-price neighbourhood group: Bronx
Highest median-price room type: Entire home/apt
Lowest median-price room type: Shared room


### Deep Dive Zero Availability

In [27]:
zero_availability_summary = (
    df_clean.assign(
        zero_availability=df_clean["availability_365"].eq(0)
    )
    .groupby("neighbourhood_group", observed=True)
    .agg(
        listing_count=("id", "count"),
        zero_availability_count=("zero_availability", "sum"),
        zero_availability_percentage=(
            "zero_availability",
            "mean",
        ),
    )
)

zero_availability_summary[
    "zero_availability_percentage"
] *= 100

zero_availability_summary.round(2)

,listing_count,zero_availability_count,zero_availability_percentage
neighbourhood_group,,,
Bronx,1090,177,16.24
Brooklyn,20095,7842,39.02
Manhattan,21660,8101,37.40
Queens,5666,1368,24.14
Staten Island,373,42,11.26


In [28]:
zero_availability_count = df_clean["availability_365"].eq(0).sum()

zero_availability_percentage = (
    df_clean["availability_365"].eq(0).mean() * 100
)

print(f"Listings with zero availability: {zero_availability_count:,}")
print(
    f"Percentage with zero availability: "
    f"{zero_availability_percentage:.2f}%"
)

Listings with zero availability: 17,530
Percentage with zero availability: 35.86%


### Deep Dive Mean vs Median

In [29]:
price_99th_percentile = df_clean["price"].quantile(0.99)

print(
    f"99th percentile of price: "
    f"${price_99th_percentile:,.2f}"
)

99th percentile of price: $799.00


In [30]:
df_price_below_99 = df_clean.loc[
    df_clean["price"].le(price_99th_percentile)
]

In [31]:
price_sensitivity = pd.DataFrame({
    "full_dataset": {
        "listing_count": len(df_clean),
        "mean_price": df_clean["price"].mean(),
        "median_price": df_clean["price"].median(),
    },
    "price_at_or_below_99th_percentile": {
        "listing_count": len(df_price_below_99),
        "mean_price": df_price_below_99["price"].mean(),
        "median_price": df_price_below_99["price"].median(),
    },
}).T

price_sensitivity.round(2)

,listing_count,mean_price,median_price
full_dataset,48884.0,152.76,106.0
price_at_or_below_99th_percentile,48410.0,137.58,105.0


### Exploratory Analysis Observations

- **Manhattan** contains the largest number of listings, accounting
  for approximately **44.31%** of the cleaned dataset.
- **Manhattan** has the highest median listing price, while
  **Bronx** has the lowest.
- **Entire home/apt** is the most common room type, while having the
  highest median price.
- Mean prices are generally higher than median prices, indicating that the
  price distributions are right-skewed by a smaller number of expensive
  listings.
- Removing prices above the 99th percentile changes the mean price from
  approximately **152.76** to **137.58**, while the median changes by
  **1**. This suggests that the median is a more stable
  measure of typical listing prices.
- The strongest numerical correlation is between **number of reviews** and
  **reviews per month**, with a correlation coefficient of approximately
  **0.59**. This indicates a moderate positive association, meaning that listings
  with more total reviews also tend to have a higher monthly review rate. This
  association should not be interpreted as causal.
- A more substantively interesting relationship appears between
  **calculated host listings count** and **availability over 365 days**, with a
  correlation of approximately **0.23**. This weak positive association suggests
  that listings managed by hosts with larger portfolios tend to be available for
  more days of the year. However, the relationship is modest and should not be
  interpreted as causal.
- Approximately **35.86%** of listings have zero recorded annual
  availability. This field does not reveal whether such listings were booked,
  inactive, or unavailable for another reason.